# 🎃 The evolution of text representations
### From keyword matching to meaning, on five texts about Halloween in Aalborg

**Module 2 · Session 01 · notebook 1 of 3**

How does a computer "read"? Over seventy years the answer changed four times. Each change came from a wall the
previous method ran into. We take the same five short texts and look at them through each lens in turn:
what it sees, and what it is blind to.

Run all cells (Runtime → Run all). Plotting code is folded away (*Show code*). The first run downloads two small
models (about 150 MB), which takes a minute or two. If Colab asks for access to a secret called `HF_TOKEN`, click *Cancel*: the models are public.

Contents
1. The corpus
2. Era 1 · Rules (1950s–1990s)
3. Era 2 · Sparse vectors: TF-IDF (1990s–2013)
4. Era 3 · Static word embeddings: GloVe (2013–2018)
5. Era 4 · Contextual embeddings: SBERT (2018–)
6. Era 5 · Large language models (2020–)
7. Which tool for which job?

In [ ]:
# @title Setup: libraries and chart style { display-mode: "form" }
%pip install -q gensim sentence-transformers
import re, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import sklearn, gensim, sentence_transformers, spacy
from matplotlib.colors import LinearSegmentedColormap
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api
from sentence_transformers import SentenceTransformer
from spacy import displacy
warnings.filterwarnings("ignore")

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    from spacy.cli import download
    download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

NAVY, LAV, GREY, ORANGE = "#211A52", "#C2C1CC", "#54616E", "#D2542A"
CMAP = LinearSegmentedColormap.from_list("aau", ["#FFFFFF", LAV, NAVY])
plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.edgecolor": GREY, "axes.labelcolor": GREY,
                     "xtick.color": GREY, "ytick.color": GREY, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_colwidth", None)

def heatmap(sim, title, ax=None):
    ax = ax or plt.subplots(figsize=(6.5, 5))[1]
    sns.heatmap(pd.DataFrame(sim, index=categories, columns=categories), annot=True, fmt=".2f",
                cmap=CMAP, vmin=0, vmax=1, cbar=False, ax=ax)
    ax.set_title(title, color=NAVY)
    return ax

print("scikit-learn", sklearn.__version__, "· gensim", gensim.__version__,
      "· sentence-transformers", sentence_transformers.__version__, "· spaCy", spacy.__version__)

---
# 🎃 1 · The corpus

Five short texts about autumn in Aalborg. Each one is written to test something:

| # | Text | What it tests |
|---|---|---|
| 1 | Official announcement | structured facts: dates, prices, places |
| 2 | Social media post | the same event in different words ("spooky", "big autumn celebration") |
| 3 | Historian's blog | related meaning, almost no shared words, and a negation |
| 4 | News article | the word *bank* in two meanings |
| 5 | AI-written promo | paraphrases text 1 |

In [ ]:
texts = [
    # 1: structured, for rule-based systems
    """The Aalborg Halloween Festival will take place from October 24th to October 31st, 2026.
    Events are centered around Gammeltorv square. The main parade is on Saturday, the 31st.
    Tickets cost 150 kr for adults and 75 kr for children.
    Activities include a pumpkin carving workshop at Aalborg Castle and a ghost tour starting from the Utzon Center.
    Please book online to secure your spot.""",
    # 2: casual, with synonyms
    """So excited for the spooky season in Aalborg! 🎃 My friends and I are totally going to the
    big autumn celebration in North Jutland this year. Heard there's a huge parade and loads of
    people in costumes. We're planning to carve some pumpkins and maybe check out the haunted walk
    by the fjord. It's going to be such a hygge time with all the scary decorations! #Aalborg #HalloweenVibes""",
    # 3: related meaning, negation
    """A local historian notes that the modern Halloween phenomenon in Denmark is a relatively recent adoption.
    It is not like the traditional Danish custom of Fastelavn, which occurs in February.
    However, the eerie fall festival in Aalborg attracts many visitors. This autumnal gathering,
    with its focus on gourds and ghouls, has a distinct character. One must treat this development
    with academic curiosity.""",
    # 4: context-dependent words
    """A recent news report discusses the climate's impact on autumn tourism in Aalborg.
    The unusually warm October of 2026 has been a mixed blessing. While pleasant weather draws more
    tourists to the bank of the Limfjord, traditional pumpkin patches have suffered. A local financial
    bank has even offered bridge loans to affected farmers. Experts are banking on a return to
    cooler temperatures next year.""",
    # 5: AI-generated, paraphrasing text 1
    """Welcome to an unforgettable autumn experience in Aalborg, the jewel of North Jutland.
    As golden leaves drift along the Limfjord, the city awakens with a magical glow.
    This year's Grand Autumn Gala, held from October 24th to 31st, promises a symphony of senses.
    Enjoy artisan pumpkin-spiced delights, marvel at the enchanted light parade near Aalborg's historic castle,
    and feel the cozy embrace of Danish 'hygge'.""",
]
categories = ["Announcement", "Social media", "Historian", "News", "AI promo"]
pd.DataFrame({"Category": categories, "Text": [" ".join(t.split()) for t in texts]})

Same texts, different lenses. Let's see what each era could see, and what it was blind to.

---
# 📜 2 · Era 1 · Rules (1950s–1990s)

**The year:** 1985. **The business problem:** pull prices and dates out of invoices and announcements into a
database. **The tool:** regular expressions, hand-written patterns that match a fixed text structure.

When you know exactly what you are looking for, and the text always looks the same, rules are fast, cheap and
precise.

In [ ]:
def extract_prices(text):
    """Finds patterns like '150 kr'."""
    return re.findall(r"\b(\d+)\s*kr\b", text)

def extract_dates(text):
    """Finds patterns like 'October 24th'."""
    return re.findall(r"\b(October\s\d{1,2}(?:st|nd|rd|th)?)\b", text)

for cat, text in zip(categories, texts):
    print(f"{cat:<13} prices: {extract_prices(text)}  dates: {extract_dates(text)}")

The rules work on the announcement and find nothing in the casual texts. They also miss "24th to 31st" in the
AI promo, because "31st" is not preceded by "October". Someone has to write a new rule for every new phrasing.

#### The professional upgrade: named entity recognition (NER)
A library like **spaCy** ships a trained model that tags dates, places, organisations, people and amounts
without any hand-written pattern.

In [ ]:
for ent in nlp(texts[0]).ents:
    print(f"{ent.text:<36} {ent.label_}")

displacy.render(nlp(texts[0]), style="ent", jupyter=True)

No rule written, and it finds the full date range *October 24th to October 31st, 2026*, plus *Saturday* and
*the 31st*. But look closely: *Gammeltorv* (a square) becomes a PERSON, *150* is just a number because the model
does not know *kr*, and *Aalborg Castle* is an organisation. The model learned from English news text, and a
Danish square name was never in it. A learned model is flexible, and its mistakes are less predictable than a
rule's.

#### The wall 🚧
Find all documents *about* the Halloween event. Text 1 says "Halloween Festival", text 2 "big autumn
celebration", text 3 "eerie fall festival". A rule has no idea that *spooky* and *eerie* are close. You would
need a hand-maintained dictionary of synonyms, and it never stops growing.

Rules capture *structure*, not *meaning*.

💡 *What if we ignore grammar and describe a document only by the words it contains?*

---
# 📊 3 · Era 2 · Sparse vectors: TF-IDF (1990s–2013)

**The year:** 2003. **The business problem:** a fast, relevant search engine over millions of internal documents.
**The tool:** bag-of-words and its upgrade, TF-IDF.

#### The big idea: from words to vectors
Every document becomes a long vector with one position per word in the vocabulary. Most positions are zero,
hence *sparse*. Plain **bag-of-words** just counts words, so frequent words like *the* dominate.
**TF-IDF** weights each count:

* **TF**, term frequency: how often the word appears in *this* document.
* **IDF**, inverse document frequency: how *rare* the word is across all documents.

TF × IDF is high for words that are frequent here and rare elsewhere: the keywords.

To compare two documents we use **cosine similarity**, the angle between their vectors: 1 means same
direction, 0 means nothing in common.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf.fit_transform(texts)
sim_tfidf = cosine_similarity(X_tfidf)
print("vocabulary size:", len(tfidf.vocabulary_), "· matrix shape:", X_tfidf.shape)

In [ ]:
# @title Heatmap: TF-IDF similarity { display-mode: "form" }
heatmap(sim_tfidf, "TF-IDF: a world of keywords")
plt.show()

#### What it could see
The announcement and the AI promo are the most similar pair (0.14): they share *october*, *castle*, *parade*,
*pumpkin*. Note how low even the best score is: with only a few shared keywords, TF-IDF scores stay small. Shared keywords are enough for a first generation of search engines.

#### The wall 🚧
Look at *Social media* vs *Historian*: 0.01, practically nothing in common. A human sees two texts about the same autumn festival
("spooky autumn celebration" vs "eerie fall festival"). TF-IDF sees almost no shared words.

The same problem in business terms: a search for *fourth quarter financial results* does not find the
document titled *Q4 earnings report*.

💡 *We know "autumn" and "fall" appear in similar surroundings. Could a word's meaning be learned from the
words around it?*

---
# 🧭 4 · Era 3 · Static word embeddings (2013–2018)

**The year:** 2013. **The breakthrough:** Word2Vec (Google, 2013), then GloVe (Stanford, 2014). A model reads a
huge corpus and learns a short, *dense* vector for every word, so that words used in similar contexts end up
close together. "You shall know a word by the company it keeps" (Firth, 1957).

We load **GloVe** vectors trained on Wikipedia and news: 400,000 words, 50 numbers each (about 65 MB).

In [ ]:
wv = api.load("glove-wiki-gigaword-50")

print("king - man + woman ≈", wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3))
print("closest to 'spooky':", [w for w, _ in wv.most_similar("spooky", topn=5)])
print(f"similarity spooky–eerie: {wv.similarity('spooky', 'eerie'):.2f} · autumn–fall: {wv.similarity('autumn', 'fall'):.2f}")

Arithmetic on meaning: *king − man + woman* lands next to *queen*. And *spooky* sits next to *creepy* and *scary*.

To get one vector per **document**, we average the vectors of its words. Stopwords (*the*, *is*) would pull
every document towards the same bland average, so we drop them first.

In [ ]:
def tokens(text):
    return [w for w in re.findall(r"[a-z]+", text.lower()) if w not in ENGLISH_STOP_WORDS]

def doc_vector(text):
    vectors = [wv[w] for w in tokens(text) if w in wv]
    return np.mean(vectors, axis=0)

X_glove = np.array([doc_vector(t) for t in texts])
sim_glove = cosine_similarity(X_glove)
print("document matrix shape:", X_glove.shape)

In [ ]:
# @title Heatmap: averaged GloVe similarity { display-mode: "form" }
heatmap(sim_glove, "GloVe, averaged: general meaning")
plt.show()

#### What it could see
*Social media* vs *Historian* jumps from 0.01 to about 0.84. The synonym problem is solved at the word level.

#### The wall 🚧
Look at the whole matrix: almost every pair is between 0.77 and 0.90. Averaging blurs everything into "a text
about Aalborg in autumn". Two more problems hide in there:

1. **One vector per word.** *Bank* has the same vector by the river and in the loan office.
2. **No word order.** "It is *not* like Fastelavn" averages to the same thing as "it is like Fastelavn".

Let's test the *bank* problem directly with four sentences.

In [ ]:
bank = ["We had a picnic on the bank of the Limfjord.",
        "The bank approved the loan for the farmers.",
        "We sat by the water's edge of the fjord.",
        "The lender granted credit to the farm."]
sim_bank_glove = cosine_similarity([doc_vector(s) for s in bank])
print("river bank vs water's edge :", round(float(sim_bank_glove[0, 2]), 2))
print("river bank vs lender       :", round(float(sim_bank_glove[0, 3]), 2))

GloVe cannot tell the riverside picnic apart from the loan office: both comparisons come out at about 0.6.

💡 *What if the vector for a word were computed after reading the whole sentence it sits in?*

---
# 🤖 5 · Era 4 · Contextual embeddings (2018–)

**The year:** 2018. **The breakthrough:** BERT and the transformer's *attention*: every word looks at every
other word in the sentence before its vector is fixed. Same word, different context, different vector.

**Sentence-BERT** (SBERT, 2019) trains such a model so that one vector represents a whole sentence or short
document, and similar meanings land close together. We use `all-MiniLM-L6-v2`: small (about 90 MB), fast on a
CPU, 384 numbers per text. We will use the same model in notebooks 2 and 3.

In [ ]:
sbert = SentenceTransformer("all-MiniLM-L6-v2")
X_sbert = sbert.encode(texts)
sim_sbert = cosine_similarity(X_sbert)
print("document matrix shape:", X_sbert.shape)

sim_bank_sbert = cosine_similarity(sbert.encode(bank))
print("river bank vs water's edge :", round(float(sim_bank_sbert[0, 2]), 2))
print("river bank vs lender       :", round(float(sim_bank_sbert[0, 3]), 2))

The *bank* test now separates: the riverside picnic is much closer to the water's edge than to the lender.

In [ ]:
# @title Three eras side by side { display-mode: "form" }
fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
heatmap(sim_tfidf, "Era 2 · TF-IDF (keywords)", axes[0])
heatmap(sim_glove, "Era 3 · GloVe (static meaning)", axes[1])
heatmap(sim_sbert, "Era 4 · SBERT (contextual meaning)", axes[2])
for ax in axes[1:]:
    ax.set_yticklabels([])
plt.tight_layout()
plt.show()

#### What it could see
* The four texts about the festival (1, 2, 3, 5) form a group, roughly 0.6 to 0.7.
* The news article, which is about weather, tourism and loans, is the least similar to the announcement
  (about 0.5).
* Unlike GloVe, the scores spread out again, so differences become visible.

Do not over-read five texts: the numbers move a little with the model version. The pattern is the point.

#### The (smaller) wall 🚧
Embeddings are for **analysis**: search, classify, cluster. They turn text into vectors; they cannot write
a sentence back.

---
# 🚀 6 · Era 5 · Large language models (2020–)

**The breakthrough:** GPT-3 (2020) and its successors. Transformer models trained to predict the next word on a
large part of the internet. Earlier eras go **text → vector**. LLMs also go **vector → text**: summarise,
answer, rewrite, classify from an instruction alone.

We work with LLMs from code in session 2. Here, one small test of what they do well and where you still have
to check them.

### ✏️ Try it: ask, then check
The cell below builds a prompt from our five texts. Copy it into any chat assistant you use (ChatGPT, Gemini,
Claude, Copilot). Then check every fact in the answer against the texts above.

In [ ]:
corpus = "\n\n".join(f"Text {i+1} ({c}):\n{' '.join(t.split())}" for i, (c, t) in enumerate(zip(categories, texts)))
prompt = f"""Here are five short texts.

{corpus}

1. Summarise them in one paragraph (max 80 words) for a tourist.
2. What does an adult ticket cost, and on which day is the main parade?
3. Which text is about something other than the festival, and what is it about?
4. Classify each text as one of: event information, personal opinion, history, news, advertising."""
print(prompt)

**Check the answer:**
* Does the summary contain anything that is *not* in the texts (a time, a place, an activity)?
* Is the parade on Saturday the 31st, as text 1 says? Did it mix up the 24th–31st date range?
* Text 5 mentions an "enchanted light parade". Did the model merge it with the main parade?

This is the habit for the rest of the module: an LLM's answer is a draft to verify, not a result.

---
# 🧰 7 · Which tool for which job?

| Business task | Good first choice | Why | Era |
|---|---|---|---|
| Extract invoice numbers, IDs, prices in a fixed format | Rules (regex) or NER | exact, fast, free | 1 |
| Keyword search, filtering reviews by product names | TF-IDF / BM25 | fast, scalable, explainable | 2 |
| Sentiment or category with many labelled examples | TF-IDF + logistic regression | strong, cheap baseline | 2 |
| Find similar documents despite different wording | Sentence embeddings | handles synonyms and paraphrase | 4 |
| Discover themes in unlabelled text | Embeddings + clustering (BERTopic) | no labels needed | 4 |
| Q&A over an internal wiki | Embeddings for retrieval + LLM for the answer (RAG) | find, then write | 4 + 5 |
| A classifier with no labelled data yet | LLM, zero-shot | prototype in an afternoon | 5 |

**The playbook:** start with the simplest method that could work. Move to embeddings or an LLM only when you
can show that the simple method falls short of what the task needs.

Each era was born from the frustration with the previous one:
rules were brittle → TF-IDF could not handle synonyms → static embeddings ignored context → contextual
embeddings could read but not write → LLMs. Modern systems use all of them together.

**Next:** notebook 2 puts TF-IDF and SBERT head to head on a real classification task.